In [ ]:
import os
from IPython.core.pylabtools import figsize

os.environ["PROJ_LIB"] = "/home/abernard/miniconda3/envs/geo_clean/share/proj"
os.environ["PROJ_DATA"] = "/home/abernard/miniconda3/envs/geo_clean/share/proj"
os.environ["GDAL_DATA"] = "/home/abernard/miniconda3/envs/geo_clean/share/gdal"

In [ ]:
import teledetection # old : import dinamis_sdk
import pystac_client
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
from path import Path
import pystac_client
import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)
from path import Path
#import pystac_client
import stackstac
import time
import planetary_computer as pc
#from simplestac.utils import ItemCollection, apply_formula, drop_assets_without_proj


In [ ]:
import warnings
from rasterio import logging


In [ ]:
from pyproj import CRS

print(CRS.from_epsg(2154))

In [ ]:
from matplotlib.colors import LogNorm

In [ ]:
# S1 images, we need to use radar veg index
#https://documentation.dataspace.copernicus.eu/APIs/openEO/openeo-community-examples/python/RVI/RVI.html

In [ ]:
gdf = gpd.read_file("../data/RNN_limite_poly_l93.shp")

gdf = gdf.set_crs("EPSG:2154")

gdf = gdf.to_crs("EPSG:4326")


gdf2 = gpd.read_file("../data/limite_RI_wgs84.shp")
gdf2 = gdf2.set_crs("EPSG:4326")


In [ ]:
#virer les warnings de rasterio
warnings.filterwarnings("ignore")
log = logging.getLogger()
log.setLevel(logging.ERROR)

In [ ]:
# Load the area of interest
roi_file = "../data/emprise_massane.geojson"
# Lire le fichier GeoJSON
#roi = gpd.read_file(roi_file)
#print(roi.total_bounds)
roi = gdf

In [ ]:
URL = "https://stacapi-cdos.apps.okd.crocc.meso.umontpellier.fr"
api = pystac_client.Client.open(URL)

collections = [(collection.id, collection.title) for collection in api.get_collections()]
pd.set_option('display.max_rows', 150)
pd.DataFrame(collections, columns=['Collection ID', 'Collection Title']).sort_values(by='Collection ID')

In [ ]:
tile = "MGRS-31TEH"
start_date = "2025-01-01"
end_date = "2026-06-01"
time_range = start_date + "/" + end_date
#time_range = "2024-03-01/2024-03-20"
#cloud_nb = 10
# bbox=[3.00, 42.47, 3.04, 42.5]

# file where the collection will be saved

catalog = pystac_client.Client.open(URL, modifier=teledetection.sign_inplace)
search = catalog.search(
    collections=["sentinel1-rtc"],  ##CDS
    #collections=["sentinel-2-l2a"], ##Planetary
    # bbox=bbox,
    bbox=roi.to_crs(4326).total_bounds,
    datetime=time_range,
    #query={"eo:cloud_cover": {"lt": cloud_nb}},
    query={"grid:code": {"eq": tile}}
    #query={"s2:mgrs_tile": {"eq": tile},"eo:cloud_cover": {"lt": cloud_nb} },
)
items = search.get_all_items()
print(f"{len(items)} items found")

In [ ]:
from simplestac.utils import ItemCollection

df = gpd.GeoDataFrame.from_features(items.to_dict(), crs="epsg:4326")

In [ ]:
col = ItemCollection(search.item_collection(), clone_items=False).sort_items(by="datetime")
item = col[0]
print(item.assets.keys())

In [ ]:
col.drop_non_raster(inplace=True)
col_non_corrected = col.filter(assets=["VH", "VV", "LIA"]) #CDS_MTD

In [ ]:
col_non_corrected

## we correct for the incidence angle (for now, N = 1)

In [ ]:
# correct the incidence angle, and convert to xarray
from simplestac.utils import apply_formula

col_non_corrected.drop_non_raster(inplace=True)

col_non_corrected.apply_items(
    fun=apply_formula, # a function that returns one or more xarray.DataArray
    name="VVnorm1",
    formula="VV*((np.cos(np.deg2rad(45))/np.cos(np.deg2rad(LIA/100)))**1)", #VVnorm
    output_dir="VVnorm1",
    #datetime="2018-01-01/..",
    geometry=roi.geometry,
    inplace=True
)
col_non_corrected.apply_items(
    fun=apply_formula, # a function that returns one or more xarray.DataArray
    name="VHnorm1",
    formula="VH*((np.cos(np.deg2rad(45))/np.cos(np.deg2rad(LIA/100)))**1)", #VHnorm
    output_dir="VHnorm1",
    #datetime="2018-01-01/..",
    geometry=roi.geometry,
    inplace=True
)

col_non_corrected.apply_items(
    fun=apply_formula, # a function that returns one or more xarray.DataArray
    name="VVnorm1.5",
    formula="VV*((np.cos(np.deg2rad(45))/np.cos(np.deg2rad(LIA/100)))**1.5)", #VVnorm
    output_dir="VVnorm15",
    #datetime="2018-01-01/..",
    geometry=roi.geometry,
    inplace=True
)
col_non_corrected.apply_items(
    fun=apply_formula, # a function that returns one or more xarray.DataArray
    name="VHnorm1.5",
    formula="VH*((np.cos(np.deg2rad(45))/np.cos(np.deg2rad(LIA/100)))**1.5)", #VHnorm
    output_dir="VHnorm15",
    #datetime="2018-01-01/..",
    geometry=roi.geometry,
    inplace=True
)

col_non_corrected.apply_items(
    fun=apply_formula, # a function that returns one or more xarray.DataArray
    name="VVnorm2",
    formula="VV*((np.cos(np.deg2rad(45))/np.cos(np.deg2rad(LIA/100)))**2)", #VVnorm
    output_dir="VVnorm2",
    #datetime="2018-01-01/..",
    geometry=roi.geometry,
    inplace=True
)
col_non_corrected.apply_items(
    fun=apply_formula, # a function that returns one or more xarray.DataArray
    name="VHnorm2",
    formula="VH*((np.cos(np.deg2rad(45))/np.cos(np.deg2rad(LIA/100)))**2)", #VHnorm
    output_dir="VHnorm2",
    #datetime="2018-01-01/..",
    geometry=roi.geometry,
    inplace=True
)


In [ ]:
xar_nc = col_non_corrected.to_xarray(geometry=roi.geometry, epsg=4326)
# #xar = col.to_xarray(geometry= geometrie)
xar_nc.drop_duplicates('time')

In [ ]:
# compute RVI

vvn1 = xar_nc.sel(band="VVnorm1")
vhn1 = xar_nc.sel(band="VHnorm1")

rvi1 = (4 * vhn1) / (vvn1 + vhn1)

# on force la structure band
rvi1 = rvi1.expand_dims("band")
rvi1 = rvi1.assign_coords(band=["RVI1"])



In [ ]:
vvn15 = xar_nc.sel(band="VVnorm1.5")
vhn15 = xar_nc.sel(band="VHnorm1.5")

rvi15 = (4 * vhn15) / (vvn15 + vhn15)

# on force la structure band
rvi15 = rvi15.expand_dims("band")
rvi15 = rvi15.assign_coords(band=["RVI1.5"])


In [ ]:
vvn2 = xar_nc.sel(band="VVnorm2")
vhn2 = xar_nc.sel(band="VHnorm2")

rvi2 = (4 * vhn2) / (vvn2 + vhn2)

# on force la structure band
rvi2 = rvi2.expand_dims("band")
rvi2 = rvi2.assign_coords(band=["RVI2"])

In [ ]:
# sans correction
vvn_nc = xar_nc.sel(band="VV")
vhn_nc = xar_nc.sel(band="VH")

rvi_nc = (4 * vhn_nc) / (vvn_nc + vhn_nc)

# on force la structure band
rvi_nc = rvi_nc.expand_dims("band")
rvi_nc = rvi_nc.assign_coords(band=["RVI_nc"])

In [ ]:
# vv/vh
vvn_nc = xar_nc.sel(band="VVnorm1")
vhn_nc = xar_nc.sel(band="VHnorm1")

vvvh1 = vvn_nc / vhn_nc

# on force la structure band
vvvh1 = vvvh1.expand_dims("band")
vvvh1 = vvvh1.assign_coords(band=["vvvh1"])

In [ ]:
xar_corrected = xr.concat(
    [xar_nc, rvi1, rvi15, rvi2, rvi_nc, vvvh1],
    dim="band"
)

In [ ]:
# passer en db
xar_corrected_db = 10 * np.log10(xar_corrected)

In [ ]:
print(xar_corrected_db.band.values)

In [ ]:
xar_corrected_db

In [ ]:
print(xar_corrected_db.sel(
    band="vvvh1",
    time="2025-07-06"
).isel(time=0).plot(figsize=(12,10), robust = True))

In [ ]:
img = xar_corrected_db.sel(
    band="VV",
    time="2025-07-06"
).isel(time=0).values

# Retirer les NaN
vals = img[np.isfinite(img)]

# Garder les 98 % centraux
q2, q98 = np.percentile(vals, [2, 98])

plt.hist(vals[(vals >= q2) & (vals <= q98)], bins=100)
plt.yscale("log")
plt.show()

# temporellement ?

In [ ]:
# Découpage sur la RI
xar_clipped = xar_corrected_db.rio.clip(
    gdf2.geometry,
    gdf2.crs,
    drop=True
)

In [ ]:
print(xar_clipped)

In [ ]:
# ne selectionner que les orbites ascendantes

#xar_clipped = xar_clipped.where(xar_clipped["sat:orbit_state"] == "descending", drop=True)

In [ ]:
# metadata for the df subsets

meta = pd.DataFrame({
    "time": xar_clipped.time.values,
    "orbit_state": xar_clipped["sat:orbit_state"].values,
    "platform": xar_clipped["platform"].values,
    "relative_orbit": xar_clipped["sat:relative_orbit"].values,
    "absolute_orbit": xar_clipped["sat:absolute_orbit"].values,
})

In [ ]:
xar_clipped = xar_clipped.chunk({
    "time": -1,
    "band": -1
})

In [ ]:
xar_clipped_ld = (
    xar_clipped
    .load()          # charge une fois
    .reset_coords(drop=True)
)

In [ ]:
df = (
    xar_clipped_ld
    .median(("x", "y"))
    .to_dataframe("value")
    .reset_index()
)

df = df.pivot(
    index="time",
    columns="band",
    values="value"
).reset_index()

In [ ]:
print(df.head)

In [ ]:
df = df.merge(meta, on="time")

In [ ]:
print(df)

# plots

In [ ]:
fig, axes = plt.subplots(
    4, 1,
    figsize=(15, 12),
    sharex=True
)

# -------- VV --------
vv_cols = ["VV", "VVnorm1", "VVnorm1.5", "VVnorm2"]

for c in vv_cols:
    axes[0].plot(df["time"], df[c], label=c)

axes[0].set_ylabel("VV")
axes[0].legend()
axes[0].grid(True)

# -------- VH --------
vh_cols = ["VH", "VHnorm1", "VHnorm1.5", "VHnorm2"]

for c in vh_cols:
    axes[1].plot(df["time"], df[c], label=c)

axes[1].set_ylabel("VH")
axes[1].legend()
axes[1].grid(True)

# -------- RVI --------
rvi_cols = ["RVI_nc", "RVI1", "RVI1.5", "RVI2"]

for c in rvi_cols:
    axes[2].plot(df["time"], df[c], label=c)

axes[2].set_ylabel("RVI")
axes[2].set_xlabel("Date")
axes[2].legend()
axes[2].grid(True)

# -------- VVVH --------
vvvh1_cols = ["vvvh1"]

for c in vvvh1_cols:
    axes[3].plot(df["time"], df[c], label=c)

axes[3].set_ylabel("VVVH")
axes[3].legend()
axes[3].grid(True)

plt.tight_layout()
plt.show()

In [ ]:

df_asc = df[df["orbit_state"] == "ascending"]
df_desc = df[df["orbit_state"] == "descending"]

plt.figure(figsize=(15, 5))

plt.plot(
    df_asc["time"],
    df_asc["vvvh1"],
    "o-",
    ms=4,
    label="Ascending"
)

plt.plot(
    df_desc["time"],
    df_desc["vvvh1"],
    "o-",
    ms=4,
    label="Descending"
)

plt.xlabel("Date")
plt.ylabel("VV / VH (dB)")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
df.groupby(["relative_orbit", "orbit_state"]).size()

In [ ]:
plt.figure(figsize=(15, 5))

for orb in [37, 59, 132]:
    d = df[df["relative_orbit"] == orb]
    plt.plot(d["time"], d["vvvh1"], "o-", label=f"Orbite {orb}")

plt.legend()
plt.show()

l'effet orbite est bien ce qui causait les fluctuations